In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import seaborn as sns
import os
# Library sesuai proposal
from feature_engine.encoding import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [3]:
# 1. Load Data
df = pd.read_csv('../data/hour.csv')

# Membuat salinan dataset
df_class = df.copy()
print(f"✓ Berhasil membuat salinan dataset. Jumlah baris: {df_class.shape[0]}")

✓ Berhasil membuat salinan dataset. Jumlah baris: 17379


In [16]:
# 2. Konversi 'cnt' menjadi Kelas Diskrit (Target) [cite: 58, 87]
def categorize_demand(cnt):
    if cnt < 50:
        return 'Low Demand'
    elif 50 <= cnt < 200:
        return 'Medium Demand'
    else:
        return 'High Demand'

df_class['demand_category'] = df_class['cnt'].apply(categorize_demand)

# Simpan ke CSV
output_path = "static/hasil_klasifikasi/demand_classification.csv"
df_class.to_csv(output_path, index=False)

print("✓ Preprocessing & pelabelan selesai")
print("✓ File disimpan ke:", output_path)

# Optional: preview di notebook
df_class[['cnt', 'demand_category']].head(20)

✓ Preprocessing & pelabelan selesai
✓ File disimpan ke: static/hasil_klasifikasi/demand_classification.csv


,cnt,demand_category
0,16,Low Demand
1,40,Low Demand
2,32,Low Demand
3,13,Low Demand
4,1,Low Demand
5,1,Low Demand
6,2,Low Demand
7,3,Low Demand
8,8,Low Demand
9,14,Low Demand


In [7]:
df_class[['season', 'weathersit']] = df_class[['season', 'weathersit']].astype(object)
features = ['season', 'hr', 'holiday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']
X = df_class[features]
y = df_class['demand_category']

encoder = OneHotEncoder(variables=['season', 'weathersit'])
X_encoded = encoder.fit_transform(X)
print(f"✓ Feature Engineering selesai. Jumlah fitur setelah encoding: {X_encoded.shape[1]}")

✓ Feature Engineering selesai. Jumlah fitur setelah encoding: 14


In [8]:
# 5. Split Data (Training & Testing)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 6. Pelatihan Model Decision Tree 
model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

# Melakukan prediksi pada data test (sebagai simulasi input data baru)
y_pred = model_dt.predict(X_test)

In [20]:
# 7. EVALUASI: CONFUSION MATRIX & REPORT
# Print Akurasi
accuracy = accuracy_score(y_test, y_pred)
print(f"Total Akurasi Model: {accuracy * 100:.2f}%")
accuracy_data = {
    "accuracy": round(float(accuracy), 4),
    "accuracy_percent": round(float(accuracy * 100), 2)
}

with open("static/hasil_klasifikasi/classification_accuracy.json", "w") as f:
    json.dump(accuracy_data, f, indent=4)

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=['Low Demand', 'Medium Demand', 'High Demand'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Low', 'Medium', 'High'], 
            yticklabels=['Low', 'Medium', 'High'])

plt.title("Confusion Matrix: Klasifikasi Tingkat Permintaan Sepeda", fontsize=14)
plt.xlabel("PREDIKSI (Hasil Model)", fontsize=12)
plt.ylabel("AKTUAL (Data Asli)", fontsize=12)
plt.tight_layout()
plt.savefig("static/hasil_klasifikasi/confusion_matrix.png")
plt.close()


print("\nDetail Laporan Klasifikasi:")
report_dict = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv("static/hasil_klasifikasi/classification_report.csv")


Total Akurasi Model: 78.71%

Detail Laporan Klasifikasi:


In [19]:
# 8. EKSPOR HASIL (FOLDER BARU)
print("\n--- TAHAP 5: Menyimpan Hasil ke CSV ---")
output_dir = 'static/hasil_klasifikasi'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"[LOG] Membuat folder baru di: {output_dir}")

# Menggabungkan fitur test dengan hasil prediksi untuk diekspor
output_df = X_test.copy()
output_df['Actual_Demand'] = y_test
output_df['Predicted_Demand'] = y_pred

output_file = f"{output_dir}/classification_result.csv"
output_df.to_csv(output_file, index=False)
print(f"[OK] Selesai! Hasil klasifikasi disimpan di: {output_file}")


--- TAHAP 5: Menyimpan Hasil ke CSV ---
[OK] Selesai! Hasil klasifikasi disimpan di: static/hasil_klasifikasi/classification_result.csv
